# Parcel boundaries from the SkyFi CDMX image

This Colab notebook processes the **SkyFi 2647SPLQ-1** delivery in `MyDrive/Dissertation/data/satellite-raw`. The six delivered files describe one WorldView-3 acquisition from 2 May 2026 at **35.5 cm/pixel**. The analytic TIFF has eight bands; this baseline uses the three-band **`_View-Ready.tif`** for RGB training and prediction. The large TIFF canvas is mostly NoData and contains two separated valid image patches. The notebook finds those patches, trains on image pixels using nearby 2021 cadastral polygons, and writes outputs for each patch.

The notebook embeds a **6,300-parcel subset** of the local six-district SIGCDMX cadastre as starting labels, so no additional label upload is needed. The polygons were selected by intersection with the two image-patch bounding boxes; some may fall on NoData within those boxes. If the cadastral outlines are misaligned or wrong, set `CORRECTED_LABEL_FILE` below to a corrected GeoPackage or GeoJSON in Drive before training. The image is used at its native 35.5 cm resolution. Nothing from the purchased imagery is committed to GitHub.

**Outputs:** a boundary-score GeoTIFF, predicted parcel polygons (GeoPackage and GeoJSON), and a per-polygon boundary-evidence GeoTIFF for each patch, plus a model checkpoint and run manifest in `MyDrive/Dissertation/data/output/parcel_pixel/skyfi_2647SPLQ-1`. These scores are not calibrated probabilities of legal-parcel correctness. Inspect the imagery, labels, and outputs before treating any polygon as a parcel. The notebook has not been validated against manually corrected parcel boundaries.


In [ ]:
# Run once in Colab. Restart the runtime if pip asks you to.
%pip -q install geopandas pyogrio shapely rasterio scipy scikit-image scikit-learn matplotlib segmentation-models-pytorch


## Mount Drive and inspect the delivery

Sign in to the Drive account that contains `Dissertation/data/satellite-raw`. The setup checks the expected metadata, CRS, bands, pixel size, and file names before any training. If the six files are still in a ZIP, extract them in Drive first.


In [ ]:
from pathlib import Path
import json
import shutil
import rasterio
from google.colab import drive

drive.mount('/content/drive')
INPUT_DIR = Path('/content/drive/MyDrive/Dissertation/data/satellite-raw')
OUTPUT_DIR = Path('/content/drive/MyDrive/Dissertation/data/output/parcel_pixel/skyfi_2647SPLQ-1')
STEM = 'SkyFi_2647SPLQ-1_2026-05-02_1708Z_DAY_SUPER_HIGH_Mexico-City-Mexico'
VIEW_READY = INPUT_DIR / f'{STEM}_View-Ready.tif'
METADATA_FILE = INPUT_DIR / f'{STEM}_metadata-item.json'
CORRECTED_LABEL_FILE = None  # e.g. INPUT_DIR / 'corrected_parcels.gpkg'
CORRECTED_LABEL_LAYER = None

if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f'Drive input folder missing: {INPUT_DIR}')
print('Drive files:', [p.name for p in sorted(INPUT_DIR.iterdir())])
for required in (VIEW_READY, METADATA_FILE):
    if not required.is_file():
        raise FileNotFoundError(f'Expected SkyFi delivery file missing: {required}')

item = json.loads(METADATA_FILE.read_text(encoding='utf-8'))
if item.get('properties', {}).get('proj:code') != 'EPSG:32614':
    raise ValueError('SkyFi metadata does not report EPSG:32614')
if abs(float(item['properties']['gsd']) - 0.355) > 0.005:
    raise ValueError('Unexpected ground sample distance in metadata')

# Copy the 21 MB visual COG to Colab's fast local disk for repeated tile reads.
SOURCE_IMAGE = Path('/content/skyfi_2647_view_ready.tif')
shutil.copy2(VIEW_READY, SOURCE_IMAGE)
with rasterio.open(SOURCE_IMAGE) as src:
    if src.crs.to_epsg() != 32614 or src.count != 3:
        raise ValueError(f'Expected EPSG:32614, three RGB bands; got {src.crs}, {src.count} bands')
    if abs(src.res[0] - 0.355) > 0.005 or abs(src.res[1] - 0.355) > 0.005:
        raise ValueError(f'Unexpected TIFF pixel size: {src.res}')
    print('Image:', src.width, 'x', src.height, 'pixels;', src.count, 'bands;', src.res, 'metres/pixel')
print('Acquisition:', item['properties']['datetime'])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Model and raster processing functions

The code below is embedded so the notebook runs when opened from GitHub in Colab without cloning the repository. The repository also contains the same source in `scripts/parcel_pixel_pipeline.py` for local development.


In [ ]:
"""Pixel-based parcel-boundary baseline for satellite_parcel_segmentation.ipynb.

The boundary probabilities are model scores, not calibrated probabilities that
an entire predicted parcel is correct. Review vector outputs before field use.
"""

from __future__ import annotations

import json
from pathlib import Path

import geopandas as gpd
import numpy as np
import pyogrio
import rasterio
from rasterio.features import rasterize, shapes
from rasterio.warp import transform_bounds
from rasterio.windows import Window, bounds as window_bounds, transform as window_transform
from scipy import ndimage as ndi
from shapely.geometry import box, shape
from skimage import measure, morphology
from sklearn.model_selection import GroupShuffleSplit
import torch
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp


def inspect_image(path: Path) -> dict:
    with rasterio.open(path) as src:
        if src.crs is None or not src.crs.is_projected:
            raise ValueError("Imagery needs a projected CRS in metres; reproject the GeoTIFF first.")
        from pyproj import CRS
        if any(axis.unit_name.lower() not in {"metre", "meter", "metres", "meters"}
               for axis in CRS.from_user_input(src.crs).axis_info[:2]):
            raise ValueError("Imagery CRS units must be metres.")
        if src.count < 3:
            raise ValueError("The baseline model requires at least three RGB bands.")
        if abs(src.res[0] - src.res[1]) > max(src.res) * 0.05:
            raise ValueError("Pixel width and height differ by more than 5%.")
        return {"crs": str(src.crs), "width": src.width, "height": src.height,
                "pixel_size_m": tuple(src.res), "bounds": tuple(src.bounds),
                "bands": src.count, "dtype": src.dtypes[0]}


def valid_image_components(image_path: Path, preview_max_dim=1024,
                           min_preview_pixels=20, padding_pixels=128) -> list[dict]:
    """Find separated valid-image patches on a mostly empty GeoTIFF canvas."""
    with rasterio.open(image_path) as src:
        scale = min(1.0, preview_max_dim / max(src.width, src.height))
        preview_width = max(1, int(np.ceil(src.width * scale)))
        preview_height = max(1, int(np.ceil(src.height * scale)))
        mask = src.dataset_mask(out_shape=(preview_height, preview_width)) > 0
        numbered, _ = ndi.label(mask, structure=np.ones((3, 3), dtype=np.uint8))
        result = []
        for component_id, slices in enumerate(ndi.find_objects(numbered), start=1):
            if slices is None or int((numbered[slices] == component_id).sum()) < min_preview_pixels:
                continue
            row, col = slices
            left = max(0, int(np.floor(col.start * src.width / preview_width)) - padding_pixels)
            right = min(src.width, int(np.ceil(col.stop * src.width / preview_width)) + padding_pixels)
            top = max(0, int(np.floor(row.start * src.height / preview_height)) - padding_pixels)
            bottom = min(src.height, int(np.ceil(row.stop * src.height / preview_height)) + padding_pixels)
            window = Window(left, top, right - left, bottom - top)
            result.append({"name": f"patch_{len(result) + 1:02d}", "window": window,
                           "bounds": window_bounds(window, src.transform),
                           "pixel_count": int(window.width * window.height)})
        if not result:
            raise ValueError("No valid imagery patches were found in the TIFF mask.")
        return result


def write_image_patch(image_path: Path, window: Window, destination: Path) -> None:
    """Copy a spatial window and its validity mask without loading the whole source."""
    with rasterio.open(image_path) as src:
        profile = src.profile.copy()
        profile.update(width=int(window.width), height=int(window.height),
                       transform=window_transform(window, src.transform))
        with rasterio.open(destination, "w", **profile) as dst:
            for _, target_block in dst.block_windows(1):
                source_block = Window(window.col_off + target_block.col_off,
                                      window.row_off + target_block.row_off,
                                      target_block.width, target_block.height)
                dst.write(src.read(window=source_block), window=target_block)
                dst.write_mask(src.dataset_mask(window=source_block), window=target_block)


def load_labels(image_path: Path, label_path: Path, layer: str | None = None) -> gpd.GeoDataFrame:
    info = pyogrio.read_info(str(label_path), layer=layer)
    if not info["crs"]:
        raise ValueError("Training polygons have no CRS.")
    with rasterio.open(image_path) as src:
        query_bbox = transform_bounds(src.crs, info["crs"], *src.bounds, densify_pts=21)
        image_bounds = box(*src.bounds)
        image_crs = src.crs
    labels = gpd.read_file(label_path, layer=layer, bbox=query_bbox, engine="pyogrio")
    if labels.empty:
        raise ValueError("No label polygons overlap the imagery.")
    labels = labels.to_crs(image_crs)
    labels = labels.loc[labels.geometry.notna() & ~labels.geometry.is_empty].copy()
    labels.geometry = labels.geometry.make_valid()
    labels = labels.loc[labels.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
    labels = labels.loc[labels.geometry.intersects(image_bounds)].copy()
    if labels.empty:
        raise ValueError("No usable polygon labels overlap the imagery.")
    return labels


def normalization_limits(image_path: Path) -> tuple[np.ndarray, np.ndarray]:
    with rasterio.open(image_path) as src:
        scale = min(1.0, 1024 / max(src.width, src.height))
        small = src.read([1, 2, 3], out_shape=(3, max(1, int(src.height * scale)),
                                                 max(1, int(src.width * scale))), masked=True)
    low, high = [], []
    for band in small:
        values = band.compressed()
        if len(values) == 0:
            raise ValueError("Imagery has no valid RGB pixels.")
        p2, p98 = np.percentile(values, [2, 98])
        low.append(float(p2))
        high.append(float(max(p98, p2 + 1)))
    return np.array(low, dtype=np.float32), np.array(high, dtype=np.float32)


def rgb_window(src, window: Window, low: np.ndarray, high: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    data = src.read([1, 2, 3], window=window, boundless=True, fill_value=0, masked=True)
    valid = ~np.any(np.ma.getmaskarray(data), axis=0)
    rgb = np.asarray(data.filled(0), dtype=np.float32)
    rgb = np.clip((rgb - low[:, None, None]) / (high - low)[:, None, None], 0, 1)
    rgb[:, ~valid] = 0
    return rgb, valid


def boundary_mask(labels: gpd.GeoDataFrame, src, window: Window, width_px: float = 1.5) -> np.ndarray:
    transform = window_transform(window, src.transform)
    footprint = box(*window_bounds(window, src.transform))
    nearby = labels.iloc[list(labels.sindex.query(footprint, predicate="intersects"))]
    width_m = width_px * max(src.res)
    edges = [(geom.boundary.buffer(width_m), 1) for geom in nearby.geometry]
    return rasterize(edges, out_shape=(int(window.height), int(window.width)),
                     transform=transform, fill=0, dtype="uint8", all_touched=True)


class ParcelTiles(Dataset):
    def __init__(self, image_path, windows, labels, low, high, augment=False):
        self.image_path, self.windows, self.labels = image_path, windows, labels
        self.low, self.high, self.augment = low, high, augment

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, index):
        with rasterio.open(self.image_path) as src:
            window = self.windows[index]
            rgb, valid = rgb_window(src, window, self.low, self.high)
            target = boundary_mask(self.labels, src, window)
        if self.augment:
            if np.random.rand() < 0.5:
                rgb, target, valid = rgb[:, :, ::-1], target[:, ::-1], valid[:, ::-1]
            if np.random.rand() < 0.5:
                rgb, target, valid = rgb[:, ::-1], target[::-1], valid[::-1]
        return (torch.from_numpy(rgb.copy()), torch.from_numpy(target[None].astype(np.float32).copy()),
                torch.from_numpy(valid[None].astype(np.float32).copy()))


def choose_windows(image_path: Path, labels: gpd.GeoDataFrame, tile_size=256,
                   max_tiles=400, seed=42, min_label_fraction=0.10,
                   min_valid_fraction=0.50):
    candidates = []
    with rasterio.open(image_path) as src:
        for row in range(0, src.height - tile_size + 1, tile_size):
            for col in range(0, src.width - tile_size + 1, tile_size):
                win = Window(col, row, tile_size, tile_size)
                footprint = box(*window_bounds(win, src.transform))
                nearby = labels.iloc[list(labels.sindex.query(footprint, predicate="intersects"))]
                if nearby.empty:
                    continue
                coverage = nearby.geometry.intersection(footprint).area.sum() / footprint.area
                if coverage >= min_label_fraction and (src.dataset_mask(window=win) > 0).mean() >= min_valid_fraction:
                    candidates.append(win)
    if len(candidates) < 8:
        raise ValueError("Fewer than eight labeled image tiles. Use a larger image or more labels.")
    rng = np.random.default_rng(seed)
    rng.shuffle(candidates)
    windows = candidates[:max_tiles]
    groups = np.array([(int(w.row_off) // (tile_size * 4), int(w.col_off) // (tile_size * 4))
                       for w in windows])
    group_codes = [f"{r}_{c}" for r, c in groups]
    if len(set(group_codes)) < 2:
        raise ValueError("Need at least two spatial blocks for a holdout split.")
    train_idx, val_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2,
                                                random_state=seed).split(windows, groups=group_codes))
    return [windows[i] for i in train_idx], [windows[i] for i in val_idx]


def build_model(pretrained=True):
    return smp.Unet(encoder_name="resnet34", encoder_weights="imagenet" if pretrained else None,
                    in_channels=3, classes=1)


def train(image_path: Path, labels: gpd.GeoDataFrame, output_dir: Path, epochs=8,
          tile_size=256, max_tiles=400, batch_size=8, seed=42, pretrained=True):
    torch.manual_seed(seed)
    np.random.seed(seed)
    low, high = normalization_limits(image_path)
    train_windows, val_windows = choose_windows(image_path, labels, tile_size, max_tiles, seed)
    train_loader = DataLoader(ParcelTiles(image_path, train_windows, labels, low, high, True),
                              batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(ParcelTiles(image_path, val_windows, labels, low, high),
                            batch_size=batch_size, num_workers=0)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = build_model(pretrained).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([4.0], device=device), reduction="none")
    best, history = -1.0, []
    for epoch in range(epochs):
        model.train()
        losses = []
        for rgb, target, valid in train_loader:
            rgb, target, valid = rgb.to(device), target.to(device), valid.to(device)
            optimizer.zero_grad()
            logits = model(rgb)
            loss = (loss_fn(logits, target) * valid).sum() / valid.sum().clamp_min(1)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        model.eval()
        tp = fp = fn = 0.0
        with torch.no_grad():
            for rgb, target, valid in val_loader:
                pred = (torch.sigmoid(model(rgb.to(device))) > 0.5).cpu()
                truth, usable = target > 0.5, valid > 0.5
                tp += (pred & truth & usable).sum().item()
                fp += (pred & ~truth & usable).sum().item()
                fn += (~pred & truth & usable).sum().item()
        f1 = 2 * tp / max(1, 2 * tp + fp + fn)
        history.append({"epoch": epoch + 1, "train_loss": float(np.mean(losses)), "validation_boundary_f1": f1})
        print(history[-1], flush=True)
        if f1 > best:
            best = f1
            torch.save({"model": model.state_dict(), "low": low.tolist(), "high": high.tolist(),
                        "tile_size": tile_size, "validation_boundary_f1": f1},
                       output_dir / "parcel_boundary_model.pt")
    (output_dir / "training_history.json").write_text(json.dumps(history, indent=2))
    return history


def predict(image_path: Path, checkpoint_path: Path, output_dir: Path,
            max_pixels=30_000_000) -> Path:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model = build_model(pretrained=False).to(device)
    model.load_state_dict(checkpoint["model"])
    model.eval()
    low = np.array(checkpoint["low"], dtype=np.float32)
    high = np.array(checkpoint["high"], dtype=np.float32)
    tile_size = int(checkpoint["tile_size"])
    with rasterio.open(image_path) as src:
        if src.width * src.height > max_pixels:
            raise ValueError("Image exceeds MAX_PIXELS; crop it to a study area before inference.")
        totals = np.zeros((src.height, src.width), dtype=np.float32)
        counts = np.zeros_like(totals)
        step = tile_size // 2
        rows = list(range(0, max(1, src.height - tile_size + 1), step))
        cols = list(range(0, max(1, src.width - tile_size + 1), step))
        rows.append(max(0, src.height - tile_size))
        cols.append(max(0, src.width - tile_size))
        with torch.no_grad():
            for row in sorted(set(rows)):
                for col in sorted(set(cols)):
                    win = Window(col, row, tile_size, tile_size)
                    rgb, valid = rgb_window(src, win, low, high)
                    probability = torch.sigmoid(model(torch.from_numpy(rgb[None]).to(device))).cpu().numpy()[0, 0]
                    h = min(tile_size, src.height - row)
                    w = min(tile_size, src.width - col)
                    totals[row:row+h, col:col+w] += probability[:h, :w] * valid[:h, :w]
                    counts[row:row+h, col:col+w] += valid[:h, :w]
        result = np.divide(totals, counts, out=np.full_like(totals, np.nan), where=counts > 0)
        profile = src.profile.copy()
        for key in ("photometric", "interleave", "alpha"):
            profile.pop(key, None)
        profile.update(driver="GTiff", count=1, dtype="float32", nodata=np.nan,
                       compress="deflate", predictor=3)
        path = output_dir / "boundary_probability.tif"
        with rasterio.open(path, "w", **profile) as dst:
            dst.write(result, 1)
            dst.set_band_description(1, "parcel boundary model score 0-1")
    return path


def vectorize(probability_path: Path, output_dir: Path, threshold=0.5,
              min_area_m2=20.0, max_area_m2=100_000.0) -> gpd.GeoDataFrame:
    with rasterio.open(probability_path) as src:
        probability = src.read(1)
        valid = np.isfinite(probability)
        edge = morphology.binary_closing((probability >= threshold) & valid, morphology.disk(1))
        edge = morphology.binary_dilation(edge, morphology.disk(1))
        regions = measure.label((~edge) & valid, connectivity=1).astype(np.int32)
        rows = []
        for geom_json, region_id in shapes(regions, mask=regions > 0, transform=src.transform):
            geom = shape(geom_json)
            area = geom.area
            if not (min_area_m2 <= area <= max_area_m2):
                continue
            boundary = geom.boundary.buffer(max(src.res))
            bounds = boundary.bounds
            from rasterio.windows import from_bounds
            win = from_bounds(*bounds, transform=src.transform)
            win = win.round_offsets().round_lengths()
            win = win.intersection(Window(0, 0, src.width, src.height))
            if win.width <= 0 or win.height <= 0:
                continue
            local = rasterize([(boundary, 1)], out_shape=(int(win.height), int(win.width)),
                              transform=window_transform(win, src.transform), dtype="uint8")
            window_probs = probability[int(win.row_off):int(win.row_off+win.height),
                                       int(win.col_off):int(win.col_off+win.width)]
            values = window_probs[(local == 1) & np.isfinite(window_probs)]
            rows.append({"parcel_id": f"PRED_{len(rows)+1:06d}", "area_m2": area,
                         "mean_boundary_probability": float(values.mean()) if len(values) else np.nan,
                         "boundary_threshold": threshold, "geometry": geom})
        parcels = gpd.GeoDataFrame(rows, columns=["parcel_id", "area_m2",
                                                  "mean_boundary_probability",
                                                  "boundary_threshold", "geometry"],
                                   geometry="geometry", crs=src.crs)
    if parcels.empty:
        raise ValueError("No polygon candidates survived filtering; inspect the probability raster and thresholds.")
    parcels.to_file(output_dir / "predicted_parcels.gpkg", layer="parcels", driver="GPKG")
    parcels.to_crs(4326).to_file(output_dir / "predicted_parcels.geojson", driver="GeoJSON")
    with rasterio.open(probability_path) as src:
        polygon_scores = rasterize(
            [(geom, float(score)) for geom, score in zip(parcels.geometry,
                                                         parcels.mean_boundary_probability)
             if np.isfinite(score)],
            out_shape=(src.height, src.width), transform=src.transform,
            fill=np.nan, dtype="float32")
        profile = src.profile.copy()
        with rasterio.open(output_dir / "parcel_boundary_evidence.tif", "w", **profile) as dst:
            dst.write(polygon_scores, 1)
            dst.set_band_description(1, "mean predicted boundary score for enclosing parcel")
    return parcels


## Load training labels and find valid patches

The embedded labels are a screening frame, not verified ground truth. The preview and patch bounds help identify NoData and alignment problems. Replace them with corrected labels when needed.


In [ ]:
import base64
import gzip
import matplotlib.pyplot as plt
import numpy as np

LABEL_FILE = Path('/content/skyfi_2026_parcel_labels.gpkg')
if CORRECTED_LABEL_FILE is None:
    compressed_labels = base64.b64decode((
    '...'
    ))
    LABEL_FILE.write_bytes(gzip.decompress(compressed_labels))
    LABEL_LAYER = 'parcels'
else:
    LABEL_FILE = Path(CORRECTED_LABEL_FILE)
    LABEL_LAYER = CORRECTED_LABEL_LAYER
if not LABEL_FILE.is_file():
    raise FileNotFoundError(LABEL_FILE)

image_info = inspect_image(SOURCE_IMAGE)
labels = load_labels(SOURCE_IMAGE, LABEL_FILE, LABEL_LAYER)
patches = valid_image_components(SOURCE_IMAGE)
print('Training polygons:', len(labels))
for patch in patches:
    print(patch['name'], 'bounds:', patch['bounds'], 'pixels:', patch['pixel_count'])
    if patch['pixel_count'] > 30_000_000:
        raise ValueError(f"{patch['name']} exceeds the 30-million-pixel patch limit; split it into smaller AOIs")


## Train the boundary model

The model uses spatially separated training and validation blocks. Validation F1 measures agreement with these cadastral labels, not legal-boundary accuracy. A Colab GPU runtime is recommended. If you already trained a model, set `TRAIN_MODEL = False` and point `CHECKPOINT_FILE` to it.


In [ ]:
TRAIN_MODEL = True
CHECKPOINT_FILE = OUTPUT_DIR / 'parcel_boundary_model.pt'
EPOCHS = 8
MAX_TRAIN_TILES = 400
TILE_SIZE = 256
BATCH_SIZE = 4
SEED = 42
PRETRAINED_ENCODER = True

if TRAIN_MODEL:
    history = train(SOURCE_IMAGE, labels, OUTPUT_DIR, epochs=EPOCHS,
                    tile_size=TILE_SIZE, max_tiles=MAX_TRAIN_TILES,
                    batch_size=BATCH_SIZE, seed=SEED, pretrained=PRETRAINED_ENCODER)
elif not CHECKPOINT_FILE.is_file():
    raise FileNotFoundError(CHECKPOINT_FILE)
print('Checkpoint:', CHECKPOINT_FILE)


## Predict both image patches

Each patch gets its own georeferenced rasters and polygons. A separate `patch_01` / `patch_02` output does not imply a one-to-one match with the four study clusters; that match needs the cluster polygons. The algorithm may split or merge parcels, especially where physical boundaries are not visible.


In [ ]:
BOUNDARY_THRESHOLD = 0.50
MIN_PARCEL_AREA_M2 = 20
MAX_PARCEL_AREA_M2 = 100_000
results = []
for patch in patches:
    patch_dir = OUTPUT_DIR / patch['name']
    patch_dir.mkdir(parents=True, exist_ok=True)
    patch_file = Path('/content') / f"skyfi_{patch['name']}.tif"
    write_image_patch(SOURCE_IMAGE, patch['window'], patch_file)
    probability_file = predict(patch_file, CHECKPOINT_FILE, patch_dir)
    predicted = vectorize(probability_file, patch_dir,
                          threshold=BOUNDARY_THRESHOLD,
                          min_area_m2=MIN_PARCEL_AREA_M2,
                          max_area_m2=MAX_PARCEL_AREA_M2)
    results.append({'patch': patch['name'], 'bounds': patch['bounds'],
                    'pixel_count': patch['pixel_count'],
                    'predicted_parcels': len(predicted),
                    'output_folder': str(patch_dir)})
    print(patch['name'], ':', len(predicted), 'candidate polygons in', patch_dir)


## Review and save the run record

Review both patch outputs in QGIS alongside the image and risk-cluster geometries. A high model score can correspond to a roof, road, or shadow rather than a parcel division.


In [ ]:
manifest = {
    'source_name': VIEW_READY.name,
    'source_metadata_name': METADATA_FILE.name,
    'acquisition_datetime': item['properties']['datetime'],
    'ground_sample_distance_m': item['properties']['gsd'],
    'source_image': image_info,
    'label_source': str(LABEL_FILE),
    'label_count': len(labels),
    'checkpoint': str(CHECKPOINT_FILE),
    'boundary_threshold': BOUNDARY_THRESHOLD,
    'results': results,
    'interpretation': 'Uncalibrated boundary evidence; candidate parcels require review.'
}
(OUTPUT_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str), encoding='utf-8')
print(json.dumps(results, indent=2, default=str))
print('Manifest:', OUTPUT_DIR / 'run_manifest.json')
